In [1]:
# --- Setup ---
import os
import torch
import torch.nn as nn
import torch.optim as optim
import joblib
import numpy as np
from tqdm import tqdm
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [7]:
# --- Constants ---
PKL_DIR = "/x1/ljiao/Bhanu/VIBE/cAvatar/dance"
FRAMES_ROOT = "/x1/ljiao/Bhanu/VIBE/output/dance"
BATCH_SIZE = 32
EPOCHS = 20
IMG_SIZE = 32  # silhouette mask resized to 32x32 → 1024D vector

In [13]:
import os
from PIL import Image
from tqdm import tqdm

# --- Where your frame folders are ---
ROOT = "/x1/ljiao/Bhanu/VIBE/output/dance"

def generate_masks_from_frames():
    folders = [f for f in os.listdir(ROOT) if f.endswith('_frames')]
    for folder in tqdm(folders, desc="Generating masks"):
        folder_path = os.path.join(ROOT, folder)
        for fname in os.listdir(folder_path):
            if fname.endswith(".png"):
                image_path = os.path.join(folder_path, fname)
                mask_path = image_path.replace(".png", "_mask.png")

                # Load image and convert to grayscale
                img = Image.open(image_path).convert("L")

                # Thresholding: any pixel > 30 becomes white (1), else black
                binary_mask = img.point(lambda p: 255 if p > 30 else 0)

                # Save mask
                binary_mask.save(mask_path)

generate_masks_from_frames()


Generating masks:  67%|██████▋   | 285/425 [16:16<07:59,  3.43s/it]


KeyboardInterrupt: 

In [8]:
# --- Dataset Loader (Silhouette Only) ---
class SilhouetteMeshDataset(Dataset):
    def __init__(self, pkl_dir, frames_root):
        self.data = []
        self.frames_root = frames_root
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])
        pkl_files = [f for f in os.listdir(pkl_dir) if f.endswith('.pkl')]
        for file_name in tqdm(pkl_files, desc="🧠 Building dataset"):
            pkl_path = os.path.join(pkl_dir, file_name)
            base_name = file_name.replace('.pkl', '')
            try:
                data = joblib.load(pkl_path)[1]
                frame_ids = data['frame_ids']
                for idx, frame_id in enumerate(frame_ids):
                    self.data.append({
                        'verts': data['verts'][idx],
                        'joints3d': data['joints3d'][idx],
                        'mask_path': os.path.join(frames_root, base_name + f'_frames/{frame_id:06d}_mask.png')
                    })
            except Exception as e:
                print(f"Error loading {file_name}: {e}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        verts = torch.tensor(sample['verts'], dtype=torch.float32).flatten()        # (20670,)
        joints3d = torch.tensor(sample['joints3d'], dtype=torch.float32)            # (49, 3)
        mask_img = Image.open(sample['mask_path']).convert('L')
        mask_tensor = self.transform(mask_img).flatten()  # (1024,)
        return mask_tensor, verts, joints3d


In [9]:
# --- MLP Model: Silhouette → Mesh ---
class Silhouette2MeshMLP(nn.Module):
    def __init__(self, input_dim=1024):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 20670)  # Output: 6890 * 3
        )

    def forward(self, x):
        return self.model(x)

In [10]:
# --- Training Function ---
def train_model(model, dataloader, device, num_epochs=20):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.MSELoss()
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        for inputs, targets, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"✅ Epoch {epoch+1} Loss: {total_loss / len(dataloader):.6f}")

In [11]:
# --- Execute Training ---
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset = SilhouetteMeshDataset(PKL_DIR, FRAMES_ROOT)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = Silhouette2MeshMLP()
    train_model(model, dataloader, device, num_epochs=EPOCHS)

    # Save model after training
    torch.save(model.state_dict(), "silhouette2mesh_model.pth")

🧠 Building dataset:  16%|█▌        | 67/425 [00:00<00:02, 155.49it/s]

Error loading Small_short_skirt_of_rabbit_ears_COS_sister_curtilage_dance_3_jumpers_dance_26_clip1.pkl: 1


🧠 Building dataset:  44%|████▎     | 185/425 [00:01<00:01, 163.24it/s]

Error loading Ziqi_flip_jump_jump_gold_in_the_summer_please_return_to_Snapping_power_P2_vertical_screen_version_of_P1_golden_summer_Snapping_please_turn_to_jump_dance_7_clip1.pkl: 1


Epoch 1/20:   0%|          | 0/3276 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/x1/ljiao/Bhanu/VIBE/output/dance/Sin_crown_Departures_POPPING_solo_dance_9_clip1_frames/000053_mask.png'

In [12]:
import os
import joblib
from tqdm import tqdm

PKL_DIR = "/x1/ljiao/Bhanu/VIBE/cAvatar/dance"
FRAMES_ROOT = "/x1/ljiao/Bhanu/VIBE/output/dance"

missing_masks = []

pkl_files = [f for f in os.listdir(PKL_DIR) if f.endswith('.pkl')]
for file_name in tqdm(pkl_files, desc="Checking mask files"):
    pkl_path = os.path.join(PKL_DIR, file_name)
    base_name = file_name.replace('.pkl', '')
    try:
        data = joblib.load(pkl_path)[1]
        frame_ids = data['frame_ids']
        for frame_id in frame_ids:
            mask_path = os.path.join(FRAMES_ROOT, base_name + f"_frames/{frame_id:06d}_mask.png")
            if not os.path.exists(mask_path):
                missing_masks.append(mask_path)
    except Exception as e:
        print(f"Error loading {file_name}: {e}")

print(f"\n🔍 Total missing masks: {len(missing_masks)}")
if missing_masks:
    print("Here are some missing examples:")
    print("\n".join(missing_masks[:10]))


Checking mask files:  16%|█▌        | 66/425 [00:00<00:02, 160.81it/s]

Error loading Small_short_skirt_of_rabbit_ears_COS_sister_curtilage_dance_3_jumpers_dance_26_clip1.pkl: 1


Checking mask files:  45%|████▌     | 192/425 [00:01<00:01, 177.86it/s]

Error loading Ziqi_flip_jump_jump_gold_in_the_summer_please_return_to_Snapping_power_P2_vertical_screen_version_of_P1_golden_summer_Snapping_please_turn_to_jump_dance_7_clip1.pkl: 1


Checking mask files: 100%|██████████| 425/425 [00:02<00:00, 173.46it/s]


🔍 Total missing masks: 104816
Here are some missing examples:
/x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_frames/000000_mask.png
/x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_frames/000001_mask.png
/x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_frames/000002_mask.png
/x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_frames/000003_mask.png
/x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_clip1_frames/000004_mask.png
/x1/ljiao/Bhanu/VIBE/output/dance/G_I_DLE_G_I_DLE_Senorita_Senorita_Dance_Cover_Cover_by_Sol_E_Mirror_Mode_gFO_Fmy5aGEkpop_dance_12_